# Chapter 2: Euclid's approach to geometry

**Source span.** The source orientation for this standalone notebook is printed pages 20-45, corresponding to physical PDF pages 30-55, sections 2.1-2.9. The PDF was used only to set the chapter boundary, terminology, and theorem order; the explanations, examples, diagrams, code, and checks here are original.

**Chapter goal.** Track how Euclid moves between angle, congruence, and area until the Pythagorean theorem, the Thales proportionality theorem, circle-angle facts, square-root constructions, and the regular pentagon all become parts of one deductive machine.

**Computational translation guide.**

| Euclid-style object | Computational representation in this notebook | Invariant to inspect |
| --- | --- | --- |
| Parallel axiom and alternate interior angles | Labeled line arrangements and sampled transversals | angle sums equal `pi`; a unique no-intersection direction through a point |
| SAS/ASA congruence | Two labeled triangles plus a proof dependency graph | corresponding sides and angles match after the stated data are fixed |
| Equality of areas | Polygon coordinates, shear matrices, and shoelace area | determinant `1`, equal base-height products, equal polygon areas |
| Pythagorean theorem | A right triangle with projected rectangles on the hypotenuse square | `a^2 + b^2 = c^2`, and `a^2 = c*c2`, `b^2 = c*c1` |
| Thales theorem | A triangle cut by a parallel segment | equal-area triangles force matching side ratios |
| Circle-angle theorems | Chords, radii, arc samples, and angle measurements | inscribed angles over the same chord are constant; a diameter subtends a right angle |
| Regular pentagon | Constructible golden-ratio length and circle intersections | all sides equal `1`; all diagonals equal `phi` |

**Visual storyboard implemented.**

| Artifact | Concept | Library route | Inspection target |
| --- | --- | --- | --- |
| `parallel-axiom-angle-cases.png` | Euclid and Playfair parallel forms, triangle angle sum | Matplotlib, because the data are planar angle/incidence relations | which angle labels are transferred by parallel lines |
| `congruence-proof-state.png` | SAS/ASA proof states: isosceles and parallelogram diagonal | Matplotlib | which side-angle-side or angle-side-angle data trigger congruence |
| `euclid-proof-dependency-graph.png` | Deductive dependencies through the chapter | NetworkX + Matplotlib | how angle, congruence, area, and proportion feed later theorems |
| `area-shear-dissection.png` | Parallelogram-to-rectangle area equality | Matplotlib + SymPy determinant check | how adding and subtracting equal triangles protects area |
| `pythagorean-area-chain.png` | Euclid's area chain and the similar-triangle revisit | Matplotlib + numeric identities | how the hypotenuse square is partitioned by projected leg squares |
| `thales-ratio-areas.png` | Thales theorem from equal-area triangles | Matplotlib + numeric ratios | where equal heights turn area ratios into length ratios |
| `circle-angle-invariance.png` | Inscribed angles and semicircle right angles | Matplotlib + numeric angle samples | why the same chord gives the same angle on an arc |
| `regular-pentagon-construction.png` | Golden-ratio pentagon construction | Matplotlib + SymPy | how `sqrt(5)` and circle intersections produce a regular pentagon |
| `sqrt-construction-lab.html` | Applied Euclid lab for square-root construction | Plotly HTML | vary a segment `l` and watch the constructed height satisfy `h^2 = l` |


In [ ]:
from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "The-Four-Pillars-of-Geometry" and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside The-Four-Pillars-of-Geometry.")


BOOK_ROOT = find_book_root(Path.cwd()).resolve()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER_KEY = "chapter-02-euclids-approach-to-geometry"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_KEY
NOTEBOOK_NAME = "euclids-approach-to-geometry.ipynb"
SOURCE_SPAN = {"printed_pages": "20-45", "pdf_pages": "30-55", "sections": "2.1-2.9"}

import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle, Polygon as MplPolygon, Rectangle
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import (
    artifact_path,
    assert_artifacts,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_json,
    save_table,
)
from utils.euclidean import polygon_area
from utils.plotting import PALETTE, draw_line, draw_polygon, equal_aspect, plot_points, save_clean

ensure_artifact_root(ARTIFACT_ROOT)
print(f"Book root: {BOOK_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## 1. Parallels turn angle motion into an equation

The parallel axiom is the first place where Euclid's geometry commits to a specific global behavior of the plane. In one form it says that two crossed lines must meet on the side where the interior angles are too small. In the Playfair form it says that, through a point off a line, there is exactly one non-meeting line.

The computational model below is intentionally small: lines are affine equations, angles are measured from direction vectors, and the triangle angle-sum proof is drawn as angle transfer across a parallel. The inspection target is not the picture alone; it is the claim that a straight angle under the parallel line has been decomposed into the three triangle angles.

In [ ]:
def angle_between(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    value = float(np.clip(np.dot(u, v) / denom, -1.0, 1.0))
    return math.acos(value)


def add_angle_arc(ax, center, radius, theta1, theta2, label, color="red"):
    arc = Arc(center, 2 * radius, 2 * radius, angle=0, theta1=theta1, theta2=theta2,
              color=PALETTE.get(color, color), lw=1.8)
    ax.add_patch(arc)
    mid = math.radians((theta1 + theta2) / 2)
    ax.text(center[0] + 1.18 * radius * math.cos(mid),
            center[1] + 1.18 * radius * math.sin(mid),
            label, color=PALETTE.get(color, color), fontsize=9, weight="bold",
            ha="center", va="center")


def save_figure(fig, filename):
    return save_clean(fig, artifact_path(ARTIFACT_ROOT, "figures", filename), dpi=165)


def make_parallel_axiom_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    x = np.linspace(-0.4, 5.2, 100)
    ax.plot(x, 0 * x, color=PALETTE["ink"], lw=2.2)
    ax.plot(x, 1.65 + 0 * x, color=PALETTE["blue"], lw=2.2)
    ax.plot([1.2, 3.75], [-0.45, 2.25], color=PALETTE["orange"], lw=2.2)
    ax.text(5.25, 0, "L", va="center", color=PALETTE["ink"], weight="bold")
    ax.text(5.25, 1.65, "M", va="center", color=PALETTE["blue"], weight="bold")
    ax.text(3.85, 2.25, "N", va="center", color=PALETTE["orange"], weight="bold")
    add_angle_arc(ax, (1.625, 0), 0.46, 0, 46.6, "alpha", "red")
    add_angle_arc(ax, (3.18, 1.65), 0.46, 180, 226.6, "pi-alpha", "green")
    ax.text(0.1, 2.25, "parallel case: interior sum is pi", color=PALETTE["ink"], fontsize=10)
    ax.set_title("Parallel line angle transfer", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.3, 5.7)
    ax.set_ylim(-0.65, 2.55)
    equal_aspect(ax, margin=0.03)

    ax = axes[1]
    A = np.array([0.35, 0.0])
    B = np.array([4.3, 0.0])
    C = np.array([2.05, 2.25])
    draw_polygon(ax, [A, B, C], color="orange", fill=True, alpha=0.13)
    ax.plot([-0.2, 4.9], [C[1], C[1]], color=PALETTE["blue"], lw=2)
    draw_line(ax, A, C, color="ink")
    draw_line(ax, B, C, color="ink")
    add_angle_arc(ax, tuple(A), 0.48, 0, math.degrees(math.atan2(C[1] - A[1], C[0] - A[0])), "alpha", "red")
    add_angle_arc(ax, tuple(B), 0.48, 180 - math.degrees(math.atan2(C[1] - B[1], B[0] - C[0])), 180, "gamma", "green")
    add_angle_arc(ax, tuple(C), 0.48, 221, 319, "beta", "purple")
    ax.text(C[0] - 0.95, C[1] + 0.22, "alpha", color=PALETTE["red"], weight="bold")
    ax.text(C[0] + 0.85, C[1] + 0.22, "gamma", color=PALETTE["green"], weight="bold")
    ax.text(1.05, -0.4, "alpha + beta + gamma = pi", color=PALETTE["ink"], fontsize=10)
    plot_points(ax, {"A": A, "B": B, "C": C}, color="blue")
    ax.set_title("Triangle angle sum from a parallel", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.1, 5.0)
    ax.set_ylim(-0.65, 2.8)
    equal_aspect(ax, margin=0.03)
    return save_figure(fig, "parallel-axiom-angle-cases.png")


parallel_path = make_parallel_axiom_figure()
A0, B0, C0 = np.array([0.35, 0.0]), np.array([4.3, 0.0]), np.array([2.05, 2.25])
parallel_angle_check = {
    "sample_triangle_angle_sum_error": abs(
        angle_between(B0 - A0, C0 - A0)
        + angle_between(A0 - B0, C0 - B0)
        + angle_between(A0 - C0, B0 - C0)
        - math.pi
    ),
    "parallel_alternate_angle_degrees": 46.6,
}
parallel_path.relative_to(BOOK_ROOT)


## 2. Congruence is a proof state, not just a measurement

Euclid's language of figures that can be moved until they coincide is replaced here by explicit state data: a theorem is allowed to use a small list of matching sides and angles, and then it may conclude the remaining matches. SAS gives the isosceles theorem by comparing a triangle with the same triangle read in the opposite order. ASA gives the parallelogram side theorem after the diagonal creates two triangles whose alternate interior angles match.

The dependency graph is a proof scaffold. It records which earlier claims are being spent, so later uses of area and proportion are not treated as isolated formulas.

In [ ]:
def make_congruence_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.9), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    A = np.array([1.5, 2.5])
    B = np.array([0.2, 0.0])
    C = np.array([2.8, 0.0])
    draw_polygon(ax, [A, B, C], color="blue", fill=True, alpha=0.11)
    draw_line(ax, A, B, color="red", lw=3, label="AB")
    draw_line(ax, A, C, color="red", lw=3, label="AC")
    add_angle_arc(ax, tuple(B), 0.42, 0, 63, "?", "green")
    add_angle_arc(ax, tuple(C), 0.42, 117, 180, "?", "green")
    plot_points(ax, {"A": A, "B": B, "C": C}, color="blue")
    ax.text(0.08, 2.85, "SAS compares ABC with ACB", color=PALETTE["ink"], fontsize=10)
    ax.text(0.08, 2.58, "same included angle at A", color=PALETTE["ink"], fontsize=9)
    ax.set_title("Isosceles theorem as a correspondence", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.25, 3.25)
    ax.set_ylim(-0.35, 3.15)
    equal_aspect(ax, margin=0.03)

    ax = axes[1]
    A = np.array([0.0, 0.0])
    B = np.array([3.3, 0.0])
    D = np.array([0.95, 1.65])
    C = B + D
    draw_polygon(ax, [A, B, C, D], color="orange", fill=True, alpha=0.12)
    draw_line(ax, A, C, color="ink", lw=2.4, linestyle="--")
    add_angle_arc(ax, tuple(A), 0.38, 0, 60, "alpha", "red")
    add_angle_arc(ax, tuple(C), 0.38, 180, 240, "alpha", "red")
    add_angle_arc(ax, tuple(A), 0.62, 30, 60, "beta", "purple")
    add_angle_arc(ax, tuple(C), 0.62, 210, 240, "beta", "purple")
    plot_points(ax, {"A": A, "B": B, "C": C, "D": D}, color="blue")
    ax.text(1.0, -0.45, "ASA on triangles ABC and CDA", color=PALETTE["ink"], fontsize=10)
    ax.text(1.0, -0.72, "opposite parallelogram sides become equal", color=PALETTE["ink"], fontsize=9)
    ax.set_title("Parallelogram side theorem", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.4, 4.6)
    ax.set_ylim(-0.95, 2.2)
    equal_aspect(ax, margin=0.03)
    return save_figure(fig, "congruence-proof-state.png")


def make_dependency_graph():
    graph = nx.DiGraph()
    nodes = [
        "parallel axiom", "SAS", "ASA", "isosceles theorem", "parallelogram sides",
        "area common notions", "parallelogram area", "triangle area",
        "Pythagorean I", "Thales", "circle angles", "Pythagorean II",
        "sqrt construction", "regular pentagon",
    ]
    graph.add_nodes_from(nodes)
    graph.add_edges_from([
        ("parallel axiom", "ASA"), ("parallel axiom", "triangle area"),
        ("SAS", "isosceles theorem"), ("ASA", "parallelogram sides"),
        ("parallelogram sides", "parallelogram area"),
        ("area common notions", "parallelogram area"),
        ("parallelogram area", "triangle area"),
        ("triangle area", "Pythagorean I"), ("triangle area", "Thales"),
        ("isosceles theorem", "circle angles"), ("circle angles", "sqrt construction"),
        ("Thales", "Pythagorean II"), ("Pythagorean II", "sqrt construction"),
        ("sqrt construction", "regular pentagon"),
    ])
    pos = {
        "parallel axiom": (0, 3), "SAS": (0, 2), "ASA": (1.4, 3),
        "isosceles theorem": (1.4, 2), "parallelogram sides": (2.8, 3),
        "area common notions": (2.8, 1.4), "parallelogram area": (4.2, 2.6),
        "triangle area": (5.6, 2.6), "Pythagorean I": (7.0, 3.2),
        "Thales": (7.0, 2.0), "circle angles": (5.6, 1.1),
        "Pythagorean II": (8.4, 2.0), "sqrt construction": (9.8, 1.5),
        "regular pentagon": (11.2, 1.5),
    }
    colors = []
    for node in graph.nodes:
        if node in {"parallel axiom", "SAS", "ASA", "area common notions"}:
            colors.append(PALETTE["light"])
        elif "Pythagorean" in node or node in {"Thales", "regular pentagon"}:
            colors.append("#f5d6be")
        else:
            colors.append("#d8e7f2")
    fig, ax = plt.subplots(figsize=(13.2, 5.3), facecolor="#fffdf7")
    nx.draw_networkx_edges(graph, pos, ax=ax, arrows=True, arrowstyle="-|>", width=1.5,
                           edge_color=PALETTE["gray"], arrowsize=14, connectionstyle="arc3,rad=0.05")
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors, edgecolors=PALETTE["ink"],
                           linewidths=1.1, node_size=2500)
    nx.draw_networkx_labels(graph, pos, ax=ax, font_size=8.6, font_color=PALETTE["ink"])
    ax.set_title("Proof-state dependency graph for Chapter 2", color=PALETTE["ink"], weight="bold", pad=12)
    ax.axis("off")
    return save_figure(fig, "euclid-proof-dependency-graph.png"), graph


congruence_path = make_congruence_figure()
dependency_path, dependency_graph = make_dependency_graph()
congruence_check = {
    "dependency_nodes": dependency_graph.number_of_nodes(),
    "dependency_edges": dependency_graph.number_of_edges(),
    "has_path_parallel_to_thales": nx.has_path(dependency_graph, "parallel axiom", "Thales"),
    "has_path_sas_to_pentagon": nx.has_path(dependency_graph, "SAS", "regular pentagon"),
}
[congruence_path.relative_to(BOOK_ROOT), dependency_path.relative_to(BOOK_ROOT)]


In [ ]:
display_artifact(parallel_path, width=860)
display_artifact(congruence_path, width=860)
display_artifact(dependency_path, width=920)


## 3. Area equality survives a shear

Euclid's area arguments do not begin by assigning decimal area values to every region. They begin with operations on figures: add equal pieces, subtract equal pieces, and replace one triangle by another with the same base and height. A modern coordinate check can keep the same spirit by tracking an invariant.

The shear matrix below has determinant `1`, so it preserves area. The diagram also shows the older cut-and-move idea: a rectangle and a parallelogram with the same base and height differ by two congruent boundary triangles.

In [ ]:
def make_area_shear_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.8), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    base, height, shear = 4.0, 1.8, 1.25
    rect = np.array([[0, 0], [base, 0], [base, height], [0, height]])
    para = np.array([[0, 0], [base, 0], [base + shear, height], [shear, height]])
    ax.add_patch(MplPolygon(rect, closed=True, facecolor="#d8e7f2", edgecolor=PALETTE["blue"], lw=2, alpha=0.55))
    ax.add_patch(MplPolygon(para, closed=True, facecolor="#f5d6be", edgecolor=PALETTE["orange"], lw=2, alpha=0.52))
    ax.add_patch(MplPolygon([[0, 0], [shear, height], [0, height]], closed=True,
                            facecolor="#eadbf8", edgecolor=PALETTE["purple"], lw=1.7, alpha=0.55))
    ax.add_patch(MplPolygon([[base, 0], [base + shear, height], [base, height]], closed=True,
                            facecolor="#eadbf8", edgecolor=PALETTE["purple"], lw=1.7, alpha=0.55))
    ax.text(1.65, 0.83, "rectangle", color=PALETTE["blue"], weight="bold")
    ax.text(2.45, 1.52, "sheared parallelogram", color=PALETTE["orange"], weight="bold")
    ax.text(4.35, 0.95, "add", color=PALETTE["purple"], weight="bold")
    ax.text(0.17, 0.95, "subtract", color=PALETTE["purple"], weight="bold")
    ax.set_title("Same base and height", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.35, 5.65)
    ax.set_ylim(-0.25, 2.35)
    equal_aspect(ax, margin=0.02)

    ax = axes[1]
    shears = np.linspace(-1.4, 1.4, 15)
    areas = []
    for value in shears:
        pts = np.array([[0, 0], [base, 0], [base + value, height], [value, height]])
        areas.append(polygon_area(pts))
    ax.plot(shears, areas, color=PALETTE["blue"], lw=2.5, marker="o", ms=4)
    ax.axhline(base * height, color=PALETTE["red"], lw=1.8, linestyle="--", label="base x height")
    ax.set_xlabel("shear parameter")
    ax.set_ylabel("shoelace area")
    ax.legend(frameon=False)
    ax.set_title("Area invariant under horizontal shear", color=PALETTE["ink"], weight="bold")
    return save_figure(fig, "area-shear-dissection.png"), np.array(areas)


area_path, shear_areas = make_area_shear_figure()
s = sp.symbols("s")
area_check = {
    "shear_determinant": str(sp.det(sp.Matrix([[1, s], [0, 1]]))),
    "area_samples_min": float(shear_areas.min()),
    "area_samples_max": float(shear_areas.max()),
    "area_target": 7.2,
    "max_area_error": float(np.max(np.abs(shear_areas - 7.2))),
}
area_path.relative_to(BOOK_ROOT)


## 4. Two Pythagorean proofs use different invariants

The first proof route is an area chain: squares and rectangles are compared by halves of equal-area triangles. The second proof route subdivides a right triangle by the altitude to the hypotenuse; the three resulting triangles are similar, so ratios become equations. The figure below keeps the same right triangle for both routes.

The labels `c1` and `c2` are the hypotenuse projections. For a right triangle with legs `a` and `b` and hypotenuse `c`, the similar-triangle proof gives `a^2 = c*c2` and `b^2 = c*c1`; adding those equations recovers `a^2 + b^2 = c^2`.

In [ ]:
def square_on_segment(P, Q, outward_point):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    v = Q - P
    left = np.array([-v[1], v[0]])
    side = np.sign(np.cross(v, np.asarray(outward_point, dtype=float) - P))
    perp = -left if side > 0 else left
    return np.array([P, Q, Q + perp, P + perp])


def make_pythagorean_figure():
    A = np.array([0.0, 0.0])
    B = np.array([5.0, 0.0])
    D = np.array([1.8, 0.0])
    C = np.array([1.8, 2.4])
    a, b, c = np.linalg.norm(B - C), np.linalg.norm(C - A), np.linalg.norm(B - A)
    c1, c2 = np.linalg.norm(D - A), np.linalg.norm(B - D)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    ax.add_patch(MplPolygon([A, B, B + [0, -5], A + [0, -5]], closed=True,
                            facecolor="#f8ead0", edgecolor=PALETTE["ink"], lw=2, alpha=0.65))
    ax.add_patch(Rectangle((0, -5), c1, 5, facecolor="#d8e7f2", edgecolor=PALETTE["blue"], lw=1.8, alpha=0.75))
    ax.add_patch(Rectangle((c1, -5), c2, 5, facecolor="#f5d6be", edgecolor=PALETTE["orange"], lw=1.8, alpha=0.75))
    ax.plot([D[0], D[0]], [0, -5], color=PALETTE["red"], lw=2, linestyle="--")
    draw_polygon(ax, [A, B, C], color="ink", fill=False)
    draw_line(ax, C, D, color="red", linestyle="--")
    ax.add_patch(MplPolygon(square_on_segment(A, C, B), closed=True,
                            facecolor="#d8e7f2", edgecolor=PALETTE["blue"], lw=2.0, alpha=0.45))
    ax.add_patch(MplPolygon(square_on_segment(C, B, A), closed=True,
                            facecolor="#f5d6be", edgecolor=PALETTE["orange"], lw=2.0, alpha=0.45))
    plot_points(ax, {"A": A, "B": B, "C": C, "D": D}, color="blue")
    ax.text(0.25, -2.6, "b^2 = c*c1", color=PALETTE["blue"], weight="bold")
    ax.text(2.55, -2.6, "a^2 = c*c2", color=PALETTE["orange"], weight="bold")
    ax.set_title("Hypotenuse square split by the altitude", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-2.8, 7.8)
    ax.set_ylim(-5.45, 5.7)
    equal_aspect(ax, margin=0.02)

    ax = axes[1]
    labels = ["c*c1", "b^2", "c*c2", "a^2", "c^2", "a^2+b^2"]
    values = [c * c1, b * b, c * c2, a * a, c * c, a * a + b * b]
    ax.bar(labels, values, color=[PALETTE["blue"], PALETTE["blue"], PALETTE["orange"], PALETTE["orange"], PALETTE["gray"], PALETTE["red"]])
    ax.set_ylabel("area value")
    ax.set_title("Numeric area identities for the same triangle", color=PALETTE["ink"], weight="bold")
    ax.tick_params(axis="x", rotation=25)
    for idx, value in enumerate(values):
        ax.text(idx, value + 0.4, f"{value:.1f}", ha="center", color=PALETTE["ink"], fontsize=9)
    return save_figure(fig, "pythagorean-area-chain.png"), {"a": a, "b": b, "c": c, "c1": c1, "c2": c2}


pythagorean_path, pythagorean_data = make_pythagorean_figure()
a, b, c, c1, c2 = [pythagorean_data[k] for k in ["a", "b", "c", "c1", "c2"]]
pythagorean_check = {
    **{k: float(v) for k, v in pythagorean_data.items()},
    "a2_equals_c_c2_error": abs(a * a - c * c2),
    "b2_equals_c_c1_error": abs(b * b - c * c1),
    "pythagorean_error": abs(a * a + b * b - c * c),
}
pythagorean_path.relative_to(BOOK_ROOT)


## 5. Thales: proportionality from equal-area triangles

The Thales theorem in this chapter is not introduced as a coordinate formula. It is a proportionality theorem extracted from area comparisons. If `PQ` is parallel to `BC`, then triangles `PQB` and `PQC` have the same base and height, so they have equal area. Adding triangle `APQ` to both creates equal larger triangles. Then triangles sharing a height turn area ratios back into base ratios.

This is the chapter's typical move: an angle fact creates a parallel, congruence or common notions protect equality, and area converts equality into a length ratio.

In [ ]:
def make_thales_figure():
    A = np.array([1.2, 3.2])
    B = np.array([0.0, 0.0])
    C = np.array([5.2, 0.0])
    t = 0.42
    P = A + t * (B - A)
    Q = A + t * (C - A)
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    draw_polygon(ax, [A, B, C], color="ink", fill=False)
    draw_line(ax, P, Q, color="blue", lw=2.8, label="PQ parallel BC")
    ax.add_patch(MplPolygon([P, Q, B], closed=True, facecolor="#d8e7f2", edgecolor=PALETTE["blue"], lw=1.6, alpha=0.55))
    ax.add_patch(MplPolygon([P, Q, C], closed=True, facecolor="#f5d6be", edgecolor=PALETTE["orange"], lw=1.6, alpha=0.55))
    plot_points(ax, {"A": A, "B": B, "C": C, "P": P, "Q": Q}, color="blue")
    ax.text(1.45, 0.84, "area PQB", color=PALETTE["blue"], weight="bold")
    ax.text(3.15, 0.84, "area PQC", color=PALETTE["orange"], weight="bold")
    ax.set_title("Equal areas under the same parallel strip", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.45, 5.65)
    ax.set_ylim(-0.35, 3.65)
    equal_aspect(ax, margin=0.02)

    ax = axes[1]
    draw_polygon(ax, [A, B, C], color="gray", fill=False)
    ax.add_patch(MplPolygon([A, P, Q], closed=True, facecolor="#eadbf8", edgecolor=PALETTE["purple"], lw=1.5, alpha=0.65))
    ax.add_patch(MplPolygon([A, Q, B], closed=True, facecolor="#d8e7f2", edgecolor=PALETTE["blue"], lw=1.5, alpha=0.4))
    ax.add_patch(MplPolygon([A, P, C], closed=True, facecolor="#f5d6be", edgecolor=PALETTE["orange"], lw=1.5, alpha=0.4))
    draw_line(ax, P, Q, color="blue", lw=2.8)
    plot_points(ax, {"A": A, "B": B, "C": C, "P": P, "Q": Q}, color="blue")
    ratio_left = np.linalg.norm(A - P) / np.linalg.norm(P - B)
    ratio_right = np.linalg.norm(A - Q) / np.linalg.norm(Q - C)
    ax.text(0.6, -0.25, f"AP/PB = {ratio_left:.3f}", color=PALETTE["blue"], weight="bold")
    ax.text(3.25, -0.25, f"AQ/QC = {ratio_right:.3f}", color=PALETTE["orange"], weight="bold")
    ax.set_title("Area ratios become side ratios", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.45, 5.65)
    ax.set_ylim(-0.55, 3.65)
    equal_aspect(ax, margin=0.02)

    data = {
        "AP_over_PB": float(ratio_left),
        "AQ_over_QC": float(ratio_right),
        "ratio_error": float(abs(ratio_left - ratio_right)),
        "area_PQB": polygon_area([P, Q, B]),
        "area_PQC": polygon_area([P, Q, C]),
    }
    data["equal_area_error"] = abs(data["area_PQB"] - data["area_PQC"])
    return save_figure(fig, "thales-ratio-areas.png"), data


thales_path, thales_check = make_thales_figure()
thales_path.relative_to(BOOK_ROOT)


In [ ]:
display_artifact(area_path, width=880)
display_artifact(pythagorean_path, width=900)
display_artifact(thales_path, width=880)


## 6. Angles in a circle and the right-angle construction

The circle-angle theorem is one of the cleanest uses of the isosceles triangle theorem. Radii create isosceles triangles, so the angle at the center is tied to twice the inscribed angle over the same chord. Keeping the chord fixed makes the center angle fixed; therefore the inscribed angle is fixed along the same arc.

When the chord is a diameter, the center angle is a straight angle, so every point on the semicircle sees a right angle. This is the missing construction device from the area proof: a right triangle can be built from its hypotenuse by drawing a semicircle.

In [ ]:
def angle_at(P, A, B):
    return angle_between(np.asarray(A) - np.asarray(P), np.asarray(B) - np.asarray(P))


def make_circle_angle_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.3), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    O = np.array([0.0, 0.0])
    R = 2.0
    A = R * np.array([math.cos(math.radians(205)), math.sin(math.radians(205))])
    B = R * np.array([math.cos(math.radians(335)), math.sin(math.radians(335))])
    sample_points = [R * np.array([math.cos(math.radians(d)), math.sin(math.radians(d))]) for d in [70, 95, 125, 155]]
    ax = axes[0]
    ax.add_patch(Circle(O, R, fill=False, edgecolor=PALETTE["ink"], lw=2))
    draw_line(ax, O, A, color="gray", lw=1.4)
    draw_line(ax, O, B, color="gray", lw=1.4)
    draw_line(ax, A, B, color="orange", lw=2.2)
    colors = ["blue", "green", "purple", "red"]
    angles = []
    for idx, C in enumerate(sample_points):
        draw_line(ax, C, A, color=colors[idx], lw=1.2, alpha=0.85)
        draw_line(ax, C, B, color=colors[idx], lw=1.2, alpha=0.85)
        angles.append(angle_at(C, A, B))
        ax.scatter([C[0]], [C[1]], s=45, color=PALETTE[colors[idx]], zorder=4)
        ax.text(C[0] + 0.05, C[1] + 0.08, f"C{idx+1}", color=PALETTE["ink"], fontsize=9, weight="bold")
    plot_points(ax, {"A": A, "B": B, "O": O}, color="blue")
    ax.text(-2.25, 2.35, "same chord AB", color=PALETTE["orange"], weight="bold")
    ax.text(-2.25, 2.08, f"sample angle std = {np.std(angles):.2e}", color=PALETTE["ink"], fontsize=9)
    ax.set_title("Inscribed angle invariant", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-2.55, 2.55)
    ax.set_ylim(-2.35, 2.65)
    equal_aspect(ax, margin=0.02)

    ax = axes[1]
    A2 = np.array([-2.0, 0.0])
    B2 = np.array([2.0, 0.0])
    t = np.linspace(0, math.pi, 200)
    ax.plot(2 * np.cos(t), 2 * np.sin(t), color=PALETTE["ink"], lw=2)
    C2 = np.array([0.65, math.sqrt(4 - 0.65**2)])
    draw_line(ax, A2, B2, color="orange", lw=2.4, label="diameter")
    draw_line(ax, C2, A2, color="blue", lw=2)
    draw_line(ax, C2, B2, color="blue", lw=2)
    plot_points(ax, {"A": A2, "B": B2, "C": C2}, color="blue")
    ax.text(-1.7, 1.8, "angle ACB = pi/2", color=PALETTE["red"], weight="bold")
    ax.text(-1.7, 1.55, "construct a right triangle from its hypotenuse", color=PALETTE["ink"], fontsize=9)
    ax.set_title("Angle in a semicircle", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-0.3, 2.45)
    equal_aspect(ax, margin=0.02)
    check = {
        "inscribed_angles_degrees": [float(math.degrees(v)) for v in angles],
        "inscribed_angle_std": float(np.std(angles)),
        "semicircle_angle_degrees": float(math.degrees(angle_at(C2, A2, B2))),
        "semicircle_right_angle_error": abs(angle_at(C2, A2, B2) - math.pi / 2),
    }
    return save_figure(fig, "circle-angle-invariance.png"), check


circle_path, circle_check = make_circle_angle_figure()
circle_path.relative_to(BOOK_ROOT)


## 7. Construction lab: square roots and the regular pentagon

The semicircle construction supplies a practical Euclidean lab. Given a segment split into lengths `l` and `1`, erect a perpendicular at the split point until it meets the semicircle on diameter `l + 1`. The altitude length `h` satisfies `h^2 = l` by the same similar-triangle proportionality used in the second Pythagorean proof.

The regular pentagon then becomes constructible because its diagonal-to-side ratio is the positive root of `x^2 - x - 1 = 0`, namely `(1 + sqrt(5))/2`. Starting from a unit side and a constructed golden-ratio diagonal, the vertices can be placed by circle intersections.

In [ ]:
def circle_intersections(P, r, Q, s):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    v = Q - P
    d = np.linalg.norm(v)
    ex = v / d
    a = (r * r - s * s + d * d) / (2 * d)
    h = math.sqrt(max(0.0, r * r - a * a))
    base = P + a * ex
    perp = np.array([-ex[1], ex[0]])
    return base + h * perp, base - h * perp


def make_pentagon_figure():
    phi = (1 + math.sqrt(5)) / 2
    A = np.array([0.0, 0.0])
    B = np.array([1.0, 0.0])
    C = max(circle_intersections(A, phi, B, 1.0), key=lambda p: p[1])
    D = max(circle_intersections(B, phi, C, 1.0), key=lambda p: p[1])
    E = min(circle_intersections(A, 1.0, D, 1.0), key=lambda p: p[0])
    pts = [A, B, C, D, E]
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.4), facecolor="#fffdf7")
    for ax in axes:
        ax.set_facecolor("#fffdf7")
        ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.7)

    ax = axes[0]
    ell = 2.35
    center = np.array([(ell + 1) / 2, 0.0])
    radius = (ell + 1) / 2
    h = math.sqrt(ell)
    theta = np.linspace(0, math.pi, 240)
    ax.plot(center[0] + radius * np.cos(theta), radius * np.sin(theta), color=PALETTE["ink"], lw=2)
    draw_line(ax, [0, 0], [ell + 1, 0], color="orange", lw=2.5)
    draw_line(ax, [ell, 0], [ell, h], color="blue", lw=2.5)
    draw_line(ax, [0, 0], [ell, h], color="gray", lw=1.2)
    draw_line(ax, [ell + 1, 0], [ell, h], color="gray", lw=1.2)
    plot_points(ax, {"0": [0, 0], "l": [ell, 0], "l+1": [ell + 1, 0], "H": [ell, h]}, color="blue")
    ax.text(ell / 2, -0.25, "l", ha="center", color=PALETTE["orange"], weight="bold")
    ax.text(ell + 0.5, -0.25, "1", ha="center", color=PALETTE["orange"], weight="bold")
    ax.text(ell + 0.1, h / 2, "h = sqrt(l)", color=PALETTE["blue"], weight="bold")
    ax.set_title("Square-root construction from a semicircle", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-0.25, ell + 1.3)
    ax.set_ylim(-0.45, 2.1)
    equal_aspect(ax, margin=0.02)

    ax = axes[1]
    draw_polygon(ax, pts, color="blue", fill=True, alpha=0.12)
    for i in range(5):
        draw_line(ax, pts[i], pts[(i + 2) % 5], color="orange", lw=1.5, alpha=0.7)
    for center_pt, rad in [(A, phi), (B, 1.0), (B, phi), (C, 1.0)]:
        ax.add_patch(Circle(center_pt, rad, fill=False, edgecolor=PALETTE["gray"], lw=1.0, linestyle="--", alpha=0.55))
    plot_points(ax, {"A": A, "B": B, "C": C, "D": D, "E": E}, color="blue")
    ax.text(-0.45, 1.75, "side = 1", color=PALETTE["blue"], weight="bold")
    ax.text(-0.45, 1.52, "diagonal = phi", color=PALETTE["orange"], weight="bold")
    ax.set_title("Regular pentagon from unit side and golden diagonal", color=PALETTE["ink"], weight="bold")
    ax.set_xlim(-1.05, 2.05)
    ax.set_ylim(-0.55, 2.15)
    equal_aspect(ax, margin=0.02)
    sides = [float(np.linalg.norm(pts[(i + 1) % 5] - pts[i])) for i in range(5)]
    diagonals = [float(np.linalg.norm(pts[(i + 2) % 5] - pts[i])) for i in range(5)]
    check = {
        "sqrt_lab_l": ell, "sqrt_lab_h": h, "sqrt_lab_h2_minus_l": h * h - ell,
        "phi": phi, "side_lengths": sides, "diagonal_lengths": diagonals,
        "max_side_error_from_1": max(abs(v - 1.0) for v in sides),
        "max_diagonal_error_from_phi": max(abs(v - phi) for v in diagonals),
    }
    return save_figure(fig, "regular-pentagon-construction.png"), check


def make_sqrt_lab_html():
    values = np.linspace(0.25, 4.0, 16)
    fig = go.Figure()
    for idx, ell in enumerate(values):
        center = np.array([(ell + 1) / 2, 0.0])
        radius = (ell + 1) / 2
        theta = np.linspace(0, math.pi, 120)
        h = math.sqrt(float(ell))
        visible = idx == 6
        fig.add_trace(go.Scatter(x=center[0] + radius * np.cos(theta), y=radius * np.sin(theta),
                                 mode="lines", line=dict(color="#243040", width=3), name="semicircle", visible=visible))
        fig.add_trace(go.Scatter(x=[0, ell + 1], y=[0, 0], mode="lines",
                                 line=dict(color="#e07a2f", width=4), name="diameter", visible=visible))
        fig.add_trace(go.Scatter(x=[ell, ell], y=[0, h], mode="lines+markers",
                                 line=dict(color="#1f77b4", width=4), marker=dict(size=8),
                                 name="constructed height", visible=visible))
        fig.add_trace(go.Scatter(x=[0, ell, ell + 1], y=[0, 0, 0], mode="markers+text",
                                 text=["0", "l", "l+1"], textposition="bottom center",
                                 marker=dict(color="#243040", size=7), showlegend=False, visible=visible))
    steps = []
    for idx, ell in enumerate(values):
        visible = [False] * (4 * len(values))
        for j in range(4):
            visible[4 * idx + j] = True
        steps.append(dict(method="update", args=[{"visible": visible}, {"title": f"Square-root construction: l={ell:.2f}, h^2={ell:.2f}"}], label=f"{ell:.2f}"))
    fig.update_layout(
        title="Square-root construction: l=1.75, h^2=1.75",
        sliders=[dict(active=6, currentvalue={"prefix": "segment l = "}, steps=steps)],
        xaxis=dict(scaleanchor="y", scaleratio=1, range=[-0.2, 5.25], zeroline=False),
        yaxis=dict(range=[-0.35, 2.45], zeroline=False),
        plot_bgcolor="#fffdf7", paper_bgcolor="#fffdf7", font=dict(color="#243040"),
        margin=dict(l=30, r=30, t=70, b=40), showlegend=True,
    )
    path = artifact_path(ARTIFACT_ROOT, "html", "sqrt-construction-lab.html")
    fig.write_html(str(path), include_plotlyjs=True, full_html=True)
    return path


pentagon_path, pentagon_check = make_pentagon_figure()
sqrt_lab_path = make_sqrt_lab_html()
x = sp.symbols("x")
phi_exact = (1 + sp.sqrt(5)) / 2
pentagon_symbolic_check = {
    "golden_ratio_polynomial_at_phi": str(sp.simplify(phi_exact**2 - phi_exact - 1)),
    "square_of_sum_identity": str(sp.expand((sp.symbols("a") + sp.symbols("b"))**2)),
}
[pentagon_path.relative_to(BOOK_ROOT), sqrt_lab_path.relative_to(BOOK_ROOT)]


In [ ]:
display_artifact(circle_path, width=900)
display_artifact(pentagon_path, width=900)
display_artifact(sqrt_lab_path, width="100%", height=460)


## Final sanity checks

The checks below deliberately mix exact symbolic assertions, numeric geometry assertions, and artifact integrity tests. This is the notebook's guardrail against treating proof diagrams as decoration: each visual has at least one invariant that can fail if the construction logic is wrong.

In [ ]:
artifact_paths = [
    parallel_path, congruence_path, dependency_path, area_path, pythagorean_path,
    thales_path, circle_path, pentagon_path, sqrt_lab_path,
]

all_checks = {
    "source_span": SOURCE_SPAN,
    "parallel": parallel_angle_check,
    "congruence_dependency": congruence_check,
    "area_shear": area_check,
    "pythagorean": pythagorean_check,
    "thales": thales_check,
    "circle_angles": circle_check,
    "pentagon": {**pentagon_check, **pentagon_symbolic_check},
}

a_sym, b_sym, s_sym = sp.symbols("a b s")
assert sp.expand((a_sym + b_sym) ** 2 - (a_sym**2 + 2 * a_sym * b_sym + b_sym**2)) == 0
assert sp.det(sp.Matrix([[1, s_sym], [0, 1]])) == 1
assert sp.simplify(((1 + sp.sqrt(5)) / 2) ** 2 - ((1 + sp.sqrt(5)) / 2) - 1) == 0

assert parallel_angle_check["sample_triangle_angle_sum_error"] < 1e-12
assert area_check["max_area_error"] < 1e-12
assert pythagorean_check["pythagorean_error"] < 1e-12
assert pythagorean_check["a2_equals_c_c2_error"] < 1e-12
assert pythagorean_check["b2_equals_c_c1_error"] < 1e-12
assert thales_check["ratio_error"] < 1e-12
assert thales_check["equal_area_error"] < 1e-12
assert circle_check["inscribed_angle_std"] < 1e-12
assert circle_check["semicircle_right_angle_error"] < 1e-12
assert pentagon_check["max_side_error_from_1"] < 1e-12
assert pentagon_check["max_diagonal_error_from_phi"] < 1e-12
assert congruence_check["has_path_parallel_to_thales"]
assert congruence_check["has_path_sas_to_pentagon"]

assert_artifacts(artifact_paths, min_size=512)
image_summaries = [image_stats(path) for path in artifact_paths if path.suffix.lower() == ".png"]
assert all(item["pixel_std"] > 2.0 for item in image_summaries)

checks_path = save_json(all_checks, ARTIFACT_ROOT, "checks", "euclid-invariants.json")
manifest_path = save_json({
    "chapter_key": CHAPTER_KEY,
    "notebook": NOTEBOOK_NAME,
    "artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in artifact_paths],
    "checks": [checks_path.relative_to(BOOK_ROOT).as_posix()],
    "tables": ["artifacts/chapter-02-euclids-approach-to-geometry/tables/construction-lab-measurements.csv"],
}, ARTIFACT_ROOT, "checks", "artifact-manifest.json")
measurements_path = save_table([
    {"measurement": "pythagorean_a", "value": pythagorean_check["a"]},
    {"measurement": "pythagorean_b", "value": pythagorean_check["b"]},
    {"measurement": "pythagorean_c", "value": pythagorean_check["c"]},
    {"measurement": "thales_AP_over_PB", "value": thales_check["AP_over_PB"]},
    {"measurement": "thales_AQ_over_QC", "value": thales_check["AQ_over_QC"]},
    {"measurement": "pentagon_phi", "value": pentagon_check["phi"]},
    {"measurement": "sqrt_lab_h2_minus_l", "value": pentagon_check["sqrt_lab_h2_minus_l"]},
], ARTIFACT_ROOT, "tables", "construction-lab-measurements.csv")
assert_artifacts([checks_path, manifest_path, measurements_path], min_size=64)

print("Artifact count:", len(artifact_paths))
print("Max Pythagorean error:", pythagorean_check["pythagorean_error"])
print("Thales ratio error:", thales_check["ratio_error"])
print("Pentagon side error:", pentagon_check["max_side_error_from_1"])


## Takeaways

Euclid's approach is not a list of unrelated facts. The parallel axiom creates reliable angle transfer; SAS and ASA turn selected equalities into congruence; common notions let equal pieces be added and subtracted; area then proves the Pythagorean theorem and Thales theorem; circle-angle invariance supplies right-angle constructions; and square roots make the golden ratio available for a regular pentagon.

The computational lesson is that each proof move protects a small invariant. Once the invariant is named, a diagram can be inspected, a symbolic identity can be checked, and a construction can be tested without pretending that coordinates replace Euclid's reasoning.